## api 조회


### 닉네임 조회

In [6]:
import requests

def search_all_servers(nickname):
    """
    모든 서버에서 해당 닉네임으로 시작하는 유저를 검색합니다.
    """
    base_url = "https://mobingi.ngrok.io"
    # 엔드포인트: /api/search/user/{user_id}
    url = f"{base_url}/api/search/user/{nickname}"
    
    try:
        response = requests.get(url)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        return f"검색 중 오류 발생: {e}"

# 사용 예시
nickname_to_find = "히로롱"
search_results = search_all_servers(nickname_to_find)
print(search_results)

검색 중 오류 발생: 404 Client Error: Not Found for url: https://mobingi.ngrok.io/api/search/user/%ED%9E%88%EB%A1%9C%EB%A1%B1


In [4]:
import pandas as pd

def display_search_results(data):
    """
    모든 서버 검색 결과를 표 형식으로 출력합니다.
    """
    if not data or isinstance(data, str):
        print("검색 결과가 없거나 오류가 발생했습니다.")
        return

    # 리스트 형태의 데이터를 DataFrame으로 변환
    df = pd.DataFrame(data)
    
    print("\n" + "="*50)
    print(f" 검색 결과: 총 {len(df)}명의 사용자를 찾았습니다.")
    print("="*50)
    
    # 필요한 컬럼만 선택하여 출력 (API 응답 컬럼명에 맞게 조정 필요)
    # 예: user_id, server_name 등
    cols = [col for col in ['user_id', 'server_name', 'level', 'class_name'] if col in df.columns]
    print(df[cols].to_string(index=False))
    print("="*50)

# 사용 예시
search_results = search_all_servers("히로롱")
display_search_results(search_results)


 검색 결과: 총 7명의 사용자를 찾았습니다.
user_id server_name
    히로롱         아이라
    히로롱          라사
    히로롱         메이븐
    히로롱          던컨
    히로롱         칼릭스
    히로롱         데이안
    히로롱         알리사


### 서버 닉네임 상세조회

In [5]:
import requests
import json

def get_user_detail(server, nickname):
    base_url = "https://mobingi.ngrok.io"
    url = f"{base_url}/api2/detail/{server}/{nickname}"
    
    try:
        response = requests.get(url)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        return {"error": f"상세 정보 조회 중 오류 발생: {e}"}

# 사용 예시
server_name = "칼릭스"
user_nickname = "히로롱"
detail_info = get_user_detail(server_name, user_nickname)

# --- [가장 중요한 부분] JSON을 정돈하여 출력 ---
# indent=4: 4칸 들여쓰기 적용
# ensure_ascii=False: 한글이 깨지지 않고 정상 출력되도록 설정
formatted_json = json.dumps(detail_info, indent=4, ensure_ascii=False)

print(formatted_json)

{
    "comments": {
        "total": 0,
        "totalPage": -1,
        "limit": 30,
        "page": 0,
        "list": [],
        "runningTime": 4
    },
    "thumbs": {
        "thumbs_up": 0
    },
    "history": [
        {
            "level": 0,
            "class_type": 8,
            "date_time": "250919",
            "class_name": "음유시인",
            "class_en_name": "bard_1",
            "parent_class": 7
        },
        {
            "level": 0,
            "class_type": 8,
            "date_time": "250920",
            "class_name": "음유시인",
            "class_en_name": "bard_1",
            "parent_class": 7
        },
        {
            "level": 0,
            "class_type": 8,
            "date_time": "250921",
            "class_name": "음유시인",
            "class_en_name": "bard_1",
            "parent_class": 7
        },
        {
            "level": 0,
            "class_type": 8,
            "date_time": "250922",
            "class_name": "음유시인",
            

In [12]:
import pandas as pd
from datetime import datetime

def display_mabinogi_profile(data):
    if not data or 'rankList' not in data:
        print("데이터를 찾을 수 없습니다.")
        return

    # 1. 기본 헤더 정보 (첫 번째 랭킹 데이터 기준)
    main_info = data['rankList'][0]
    print("\n" + "═"*60)
    print(f" 🛡️  [{main_info['server_name']}] {main_info['user_id']} 님의 통합 프로필")
    print(f" 🏆  종합 평가 점수: {main_info['evaluation_score']:,} | 추천 수: {data['thumbs']['thumbs_up']}")
    print("═"*60)

    # 2. 클래스별 현재 상태 (rankList 항목 요약)
    print("\n [ 클래스별 현재 능력치 ]")
    rank_df = pd.DataFrame(data['rankList'])
    # 가독성을 위해 주요 컬럼만 선택 및 이름 변경
    rank_summary = rank_df[['class_name', 'level', 'rank', 'class_rank', 'vitality', 'attractiveness']].copy()
    rank_summary.columns = ['직업', '레벨', '전체순위', '직업순위', '생명력', '매력']
    # 순위 데이터가 None인 경우 '-'로 표시
    rank_summary = rank_summary.fillna('-')
    print(rank_summary.to_string(index=False))

    # 3. 최근 성장 기록 (history 항목 중 최신 10개만 출력)
    print("\n [ 최근 레벨 성장 기록 (History) ]")
    history_df = pd.DataFrame(data['history'])
    if not history_df.empty:
        # 날짜 형식 변환 및 정렬 (YYMMDD -> YYYY-MM-DD)
        history_df['date'] = history_df['date_time'].apply(lambda x: f"20{x[:2]}-{x[2:4]}-{x[4:]}")
        # 최근 10개만 추출 (가장 마지막이 최신)
        recent_history = history_df.tail(10)[['date', 'class_name', 'level']]
        recent_history.columns = ['날짜', '직업', '전투력/레벨']
        print(recent_history.to_string(index=False))
    
    # 4. 소셜 및 업데이트 정보
    print("\n" + "─"*60)
    print(f" 💬 등록된 댓글: {data['comments']['total']}개")
    print(f" 🕒 최종 업데이트: {main_info['update_at']}")
    print("─"*60 + "\n")

# 실행
display_mabinogi_profile(detail_info)


════════════════════════════════════════════════════════════
 🛡️  [칼릭스] 히로롱 님의 통합 프로필
 🏆  종합 평가 점수: 126,239 | 추천 수: 0
════════════════════════════════════════════════════════════

 [ 클래스별 현재 능력치 ]
  직업    레벨 전체순위 직업순위   생명력    매력
음유시인 48793    -    - 47784 19127
  악사 59328    0    0 47784 19127
  힐러 50041    -    - 47784 19127
화염술사 46245    -    - 47784 19127
빙결술사 47061    -    - 47784 19127
 검술사 38002    -    - 47784 19127

 [ 최근 레벨 성장 기록 (History) ]
        날짜  직업  전투력/레벨
2026-01-06 검술사   38002
2026-01-07 검술사   38002
2026-01-08 검술사   38002
2026-01-09 검술사   38002
2026-01-10 검술사   38002
2026-01-11 검술사   38002
2026-01-12 검술사   38002
2026-01-13 검술사   38002
2026-01-14 검술사   38002
2026-01-15 검술사   38002

────────────────────────────────────────────────────────────
 💬 등록된 댓글: 0개
 🕒 최종 업데이트: 2026-01-15T10:17:45.000Z
────────────────────────────────────────────────────────────



In [7]:
import json

def get_main_class_history(data):
    if not data or 'rankList' not in data:
        return "데이터가 유효하지 않습니다."

    # 1. rankList에서 enable_class가 "1"인 메인 클래스의 class_type 찾기
    main_class_info = next((item for item in data['rankList'] if item.get('enable_class') == "1"), None)
    
    if not main_class_info:
        return "메인 클래스(enable_class='1')를 찾을 수 없습니다."

    main_type = main_class_info['class_type']
    main_name = main_class_info['class_name']

    # 2. history 데이터에서 해당 class_type만 필터링
    main_history = [
        item for item in data.get('history', []) 
        if item.get('class_type') == main_type
    ]

    # 3. 가독성을 위해 정돈된 결과 생성
    result = {
        "user_id": main_class_info['user_id'],
        "main_class": main_name,
        "class_type": main_type,
        "history_count": len(main_history),
        "history": main_history
    }
    
    return result

# --- 실행 및 출력 ---
# detail_info는 이전 단계에서 호출한 전체 JSON 데이터입니다.
filtered_data = get_main_class_history(detail_info)

# 필터링된 전체 내용을 JSON 형식으로 출력
print(json.dumps(filtered_data, indent=4, ensure_ascii=False))

{
    "user_id": "히로롱",
    "main_class": "악사",
    "class_type": 10,
    "history_count": 109,
    "history": [
        {
            "level": 0,
            "class_type": 10,
            "date_time": "250919",
            "class_name": "악사",
            "class_en_name": "bard_3",
            "parent_class": 7
        },
        {
            "level": 0,
            "class_type": 10,
            "date_time": "250920",
            "class_name": "악사",
            "class_en_name": "bard_3",
            "parent_class": 7
        },
        {
            "level": 32981,
            "class_type": 10,
            "date_time": "250921",
            "class_name": "악사",
            "class_en_name": "bard_3",
            "parent_class": 7
        },
        {
            "level": 32981,
            "class_type": 10,
            "date_time": "250922",
            "class_name": "악사",
            "class_en_name": "bard_3",
            "parent_class": 7
        },
        {
            "level": 3298

In [8]:
import json
from datetime import datetime

def display_user_summary(data):
    # 1. 메인 클래스 특정 (enable_class == "1")
    main_class = next((item for item in data['rankList'] if item.get('enable_class') == "1"), None)
    
    if not main_class:
        print("메인 클래스 정보를 찾을 수 없습니다.")
        return

    # 2. 날짜 포맷 변경 (ISO 8601 -> YYYY-MM-DD HH:MM)
    raw_date = main_class.get('update_at')
    if raw_date:
        try:
            # ISO 8601 포맷 파싱
            formatted_date = datetime.strptime(raw_date, "%Y-%m-%dT%H:%M:%S.%fZ").strftime("%Y-%m-%d %H:%M")
        except ValueError:
            formatted_date = raw_date # 파싱 실패 시 원본 표시
    else:
        formatted_date = "정보 없음"

    # 3. 메인 클래스의 히스토리 필터링 및 변화량 계산
    main_type = main_class['class_type']
    # 메인 클래스 기록만 추출
    history_list = [h for h in data.get('history', []) if h.get('class_type') == main_type]
    
    # 변화량 계산을 위해 최근 8개 추출 (7일치 변화량을 보려면 8일치 데이터가 필요함)
    recent_8_days = history_list[-8:]
    
    display_history = []
    # 뒤에서부터 순회하며 변화량 계산
    for i in range(len(recent_8_days) - 1, 0, -1):
        curr = recent_8_days[i]
        prev = recent_8_days[i-1]
        
        diff = curr['level'] - prev['level']
        diff_str = f"(+{diff:,})" if diff > 0 else f"({diff:,})" if diff < 0 else "(+0)"
        
        d = curr['date_time']
        date_str = f"20{d[:2]}-{d[2:4]}-{d[4:]}"
        
        display_history.append(f"[{date_str}] 전투력: {curr['level']:,} {diff_str}")

    # --- 출력부 ---
    print(f"닉네임: {main_class['user_id']}")
    print(f"서버: {main_class['server_name']}")
    print(f"직업: {main_class['class_name']} (메인)")
    print(f"서버내 순위: {main_class.get('server_rank', '-')}")
    print(f"직업 순위: {main_class.get('class_rank', '-')}")
    print("")
    print(f"종합 평가: {main_class.get('evaluation_score', 0):,}")
    print(f"전투력: {main_class.get('level', 0):,}")
    print(f"매력: {main_class.get('vitality', 0):,}")
    print(f"생활력: {main_class.get('attractiveness', 0):,}")
    print("")
    print(f"마지막 접속: {formatted_date}")
    print("")
    print("최근 성장 추이 (최신순 7일)")
    print("-" * 40)
    for line in display_history[:7]: # 최신 7개만 출력
        print(line)
    print("-" * 40)

# 실행
display_user_summary(detail_info)

닉네임: 히로롱
서버: 칼릭스
직업: 악사 (메인)
서버내 순위: 1657
직업 순위: 0

종합 평가: 126,239
전투력: 59,328
매력: 47,784
생활력: 19,127

마지막 접속: 2026-01-15 10:17

최근 성장 추이 (최신순 7일)
----------------------------------------
[2026-01-15] 전투력: 58,924 (+53)
[2026-01-14] 전투력: 58,871 (+60)
[2026-01-13] 전투력: 58,811 (+50)
[2026-01-12] 전투력: 58,761 (+4)
[2026-01-11] 전투력: 58,757 (+74)
[2026-01-10] 전투력: 58,683 (+0)
[2026-01-09] 전투력: 58,683 (+0)
----------------------------------------
